# A2A — agents that talk to each other over the network

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhelamay/CPUT_ICAILY_AI2026_MaganLLM_WORKSHOP/blob/main/SA_LIB_WORKSHOP_Day2/07_a2a_multi_agent/07_a2a_multi_agent.ipynb)

**SA-LIB AI Workshop 2026 — Day 2: AI Agents**

The **Agent2Agent (A2A)** protocol lets agents built with *any* framework discover and call each other over HTTP. Two agents run as separate web services (researcher on :8001, writer on :8002); a plain A2A client and an ADK orchestrator use them.

Same example in every framework: **(1) first agent → (2) tools → (3) memory → (4) fallback → (5) multi-agent → complete agent**.

## 0. Setup
Install the packages (takes ~1 minute in Colab).

In [ ]:
%pip install -q "google-adk[a2a]>=1.0" "litellm" "uvicorn" "python-dotenv" "ddgs"

### API keys
* **Colab:** click the 🔑 *Secrets* icon on the left, add e.g. `GROQ_API_KEY`, and enable notebook access.
* **VS Code / Jupyter / Linux:** put your keys in the `.env` file in the repo root (copy `.env.example`).
* Otherwise you'll be asked to paste a key below.

Choose the model provider with `LLM_PROVIDER`: `groq` · `gemini` · `openai` · `deepseek` · `openrouter` · `ollama`.

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))          # VS Code / Linux: read ../.env
try:                                           # Colab: read the Secrets panel
    from google.colab import userdata
    for key in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'DEEPSEEK_API_KEY', 'OPENROUTER_API_KEY', 'LANGSMITH_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST', 'LLM_PROVIDER', 'LLM_MODEL']:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("LLM_PROVIDER", "groq")   # <- change the provider here if you like
provider = os.environ["LLM_PROVIDER"]
key_name = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY",
            "deepseek": "DEEPSEEK_API_KEY", "openrouter": "OPENROUTER_API_KEY"}.get(provider)
if key_name and not os.getenv(key_name):
    os.environ[key_name] = getpass.getpass(f"{key_name}: ")
print("provider:", provider)

### Shared helper: `workshop_common.py`
Reads the provider settings, and holds the two tools every example uses: `web_search` (free DuckDuckGo) and `word_count`.

In [ ]:
%%writefile workshop_common.py
"""
workshop_common.py — shared helpers for every framework folder in this workshop.

(the SAME file is copied into each folder so every folder works on its own —
 edit shared/workshop_common.py and run `python tools/sync_common.py` to update the copies)

What it gives you:
  get_llm_config()   -> which provider / model / base_url / api key to use (read from .env)
  fallback_order()   -> the list of providers to try, in order, for the fallback examples
  ping()             -> quick "are you alive?" check against a provider
  web_search()       -> free DuckDuckGo search (no key)
  word_count()       -> the tiny "tool you wrote yourself" from the original guide

Every provider below speaks the OpenAI chat-completions protocol, which is why every
framework in this workshop can use every provider: we just change base_url + api_key + model.
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from dotenv import find_dotenv, load_dotenv

# walks up from where you run python, so the .env in the repo root is always found
load_dotenv(find_dotenv(usecwd=True))

PROVIDERS = {
    #  name        base_url                                                     api-key env var       default model
    "groq":       ("https://api.groq.com/openai/v1",                           "GROQ_API_KEY",       "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY",     "gemini-2.5-flash"),
    "openai":     ("https://api.openai.com/v1",                                "OPENAI_API_KEY",     "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com",                                 "DEEPSEEK_API_KEY",   "deepseek-chat"),
    "openrouter": ("https://openrouter.ai/api/v1",                             "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),  None,                 "qwen2.5:7b"),
}


@dataclass
class LLMConfig:
    provider: str
    model: str
    base_url: str
    api_key: str

    @property
    def litellm_model(self) -> str:
        """Model name in LiteLLM format (used by CrewAI and Google ADK): 'openai/<model>'
        means 'an OpenAI-compatible endpoint' — combined with base_url it works for every provider."""
        return f"openai/{self.model}"


def get_llm_config(provider: str | None = None) -> LLMConfig:
    """Read LLM_PROVIDER / LLM_MODEL / <PROVIDER>_API_KEY from the environment."""
    provider = (provider or os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown LLM_PROVIDER '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    base_url, key_env, default_model = PROVIDERS[provider]
    api_key = os.getenv(key_env) if key_env else "ollama"  # ollama runs locally, no key
    if not api_key:
        raise RuntimeError(f"{key_env} is missing — add it to your .env file (see .env.example).")
    # LLM_MODEL only overrides the model of the MAIN provider, not of the fallbacks
    main_provider = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    model = (os.getenv("LLM_MODEL") if provider == main_provider else None) or default_model
    return LLMConfig(provider, model, base_url, api_key)


def fallback_order() -> list[str]:
    """Providers to try, in order. Default: your LLM_PROVIDER first, then LLM_FALLBACKS (groq,gemini)."""
    main = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    extra = [p.strip().lower() for p in os.getenv("LLM_FALLBACKS", "groq,gemini").split(",") if p.strip()]
    return list(dict.fromkeys([main, *extra]))  # remove duplicates, keep order


def ping(cfg: LLMConfig) -> None:
    """Send a 1-token request. Raises an exception if the provider is down, rate-limited or the key is wrong."""
    from openai import OpenAI

    OpenAI(base_url=cfg.base_url, api_key=cfg.api_key, timeout=20, max_retries=0).chat.completions.create(
        model=cfg.model, messages=[{"role": "user", "content": "ping"}], max_tokens=1
    )


def web_search(query: str) -> str:
    """Search the web with DuckDuckGo (free, no key) and return the top results as text."""
    try:
        from ddgs import DDGS

        results = DDGS().text(query, max_results=3)
        return "\n".join(f"- {r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:  # the free search is rate-limited: return the error instead of crashing
        return f"web search failed ({e}). try again in a few seconds."


def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

### helper file: `common_model.py`
picks the model for the ADK agents in this folder (gemini natively, everything else via LiteLLM)

In [ ]:
%%writefile common_model.py
# picks the model for the ADK agents in this folder (gemini natively, everything else via LiteLLM)
from google.adk.models.lite_llm import LiteLlm

from workshop_common import get_llm_config


def adk_model():
    cfg = get_llm_config()
    if cfg.provider == "gemini":
        return cfg.model
    return LiteLlm(model=cfg.litellm_model, api_base=cfg.base_url, api_key=cfg.api_key)

### helper file: `researcher_server.py`
A2A agent #1 — the RESEARCHER, running as its own web service on port 8001.
to_a2a() wraps any ADK agent in an A2A server and publishes its "business card" at
    http://localhost:8001/.well-known/agent-card.json
(name, description, skills, how to talk to it). any A2A client, in any framework
or language, can now discover and call this agent.

In [ ]:
%%writefile researcher_server.py
# A2A agent #1 — the RESEARCHER, running as its own web service on port 8001.
# to_a2a() wraps any ADK agent in an A2A server and publishes its "business card" at
#     http://localhost:8001/.well-known/agent-card.json
# (name, description, skills, how to talk to it). any A2A client, in any framework
# or language, can now discover and call this agent.
#
# run (terminal 1):  python 07_a2a_multi_agent/researcher_server.py

import uvicorn
from google.adk.agents import Agent
from google.adk.a2a.utils.agent_to_a2a import to_a2a

from common_model import adk_model
from workshop_common import web_search

PORT = 8001

researcher = Agent(
    name="researcher",
    description="Finds current facts on the web and returns 3 short bullet points with sources.",
    model=adk_model(),
    instruction="Use web_search to find current facts about the user's topic. Reply with 3 short bullet facts.",
    tools=[web_search],
)

app = to_a2a(researcher, host="localhost", port=PORT)

if __name__ == "__main__":
    uvicorn.run(app, host="localhost", port=PORT)

### helper file: `writer_server.py`
A2A agent #2 — the WRITER, running as its own web service on port 8002.
it knows nothing about the researcher. the orchestrator connects them over A2A.

In [ ]:
%%writefile writer_server.py
# A2A agent #2 — the WRITER, running as its own web service on port 8002.
# it knows nothing about the researcher. the orchestrator connects them over A2A.
#
# run (terminal 2):  python 07_a2a_multi_agent/writer_server.py

import uvicorn
from google.adk.agents import Agent
from google.adk.a2a.utils.agent_to_a2a import to_a2a

from common_model import adk_model
from workshop_common import word_count

PORT = 8002

writer = Agent(
    name="writer",
    description="Turns facts into one friendly paragraph under 60 words for beginners.",
    model=adk_model(),
    instruction="Write one friendly paragraph under 60 words from the facts you are given. "
                "Check the length with word_count before answering.",
    tools=[word_count],
)

app = to_a2a(writer, host="localhost", port=PORT)

if __name__ == "__main__":
    uvicorn.run(app, host="localhost", port=PORT)

### helper file: `a2a_client.py`
the A2A protocol with NO agent framework: discover an agent, send it a message, read the reply.
this is all any client (python, javascript, java, ...) needs to talk to an A2A agent.

In [ ]:
%%writefile a2a_client.py
# the A2A protocol with NO agent framework: discover an agent, send it a message, read the reply.
# this is all any client (python, javascript, java, ...) needs to talk to an A2A agent.
#
# run (terminal 3, servers must be running):  python 07_a2a_multi_agent/a2a_client.py

import asyncio

import httpx
from a2a.client import A2ACardResolver, create_client
from a2a.helpers import get_stream_response_text, new_text_message
from a2a.types import SendMessageRequest

AGENTS = {"researcher": "http://localhost:8001", "writer": "http://localhost:8002"}


async def ask(url: str, text: str) -> str:
    async with httpx.AsyncClient() as http:            # 1. DISCOVER: download the agent card
        card = await A2ACardResolver(http, base_url=url).get_agent_card()
    print(f"\n--> {card.name}: {card.description}")
    client = await create_client(card)                 # 2. CONNECT using what the card says
    request = SendMessageRequest(message=new_text_message(text))
    answer = ""
    async for response in client.send_message(request):  # 3. SEND a message, stream the replies
        answer = get_stream_response_text(response) or answer
    return answer


async def main():
    facts = await ask(AGENTS["researcher"], "the latest langchain version")
    print(facts)
    paragraph = await ask(AGENTS["writer"], f"Facts:\n{facts}")
    print(paragraph)


if __name__ == "__main__":
    asyncio.run(main())

### helper file: `orchestrator.py`
the ORCHESTRATOR — an ADK agent whose team members live in OTHER processes.
RemoteA2aAgent reads each agent card and makes the remote agent look like a local sub-agent.
SequentialAgent runs them in order: researcher -> writer.

In [ ]:
%%writefile orchestrator.py
# the ORCHESTRATOR — an ADK agent whose team members live in OTHER processes.
# RemoteA2aAgent reads each agent card and makes the remote agent look like a local sub-agent.
# SequentialAgent runs them in order: researcher -> writer.
#
# run (terminal 3, servers must be running):  python 07_a2a_multi_agent/orchestrator.py

import asyncio

from google.adk.a2a.agent import RemoteA2aAgent
from google.adk.agents import SequentialAgent
from google.adk.runners import InMemoryRunner
from google.genai import types

CARD = "/.well-known/agent-card.json"

researcher = RemoteA2aAgent(name="researcher", agent_card=f"http://localhost:8001{CARD}",
                            description="Finds current facts on the web.")
writer = RemoteA2aAgent(name="writer", agent_card=f"http://localhost:8002{CARD}",
                        description="Writes a short paragraph from facts.")

team = SequentialAgent(name="team", sub_agents=[researcher, writer])


async def main():
    runner = InMemoryRunner(agent=team, app_name="a2a_demo")
    session = await runner.session_service.create_session(app_name="a2a_demo", user_id="m0h")
    message = types.Content(role="user", parts=[types.Part(text="the latest langchain version")])
    printed = set()  # a remote agent can report the same answer twice (status + artifact): print it once
    async for event in runner.run_async(user_id="m0h", session_id=session.id, new_message=message):
        if event.is_final_response() and event.content and event.content.parts and event.content.parts[0].text:
            key = (event.author, event.content.parts[0].text)
            if key not in printed:
                printed.add(key)
                print(f"\n[{event.author}]\n{event.content.parts[0].text}")


if __name__ == "__main__":
    asyncio.run(main())

### helper file: `run_demo.py`
one command for the whole demo (handy in Colab / VS Code): starts both A2A servers
in the background, waits until their agent cards are online, runs the client + orchestrator,
then stops the servers.

In [ ]:
%%writefile run_demo.py
# one command for the whole demo (handy in Colab / VS Code): starts both A2A servers
# in the background, waits until their agent cards are online, runs the client + orchestrator,
# then stops the servers.
#
# run:  python 07_a2a_multi_agent/run_demo.py

import subprocess
import sys
import time
import urllib.request
from pathlib import Path

HERE = Path(__file__).resolve().parent
SERVERS = {"researcher_server.py": 8001, "writer_server.py": 8002}


def wait_for(port: int, timeout: int = 60) -> None:
    url = f"http://localhost:{port}/.well-known/agent-card.json"
    for _ in range(timeout):
        try:
            urllib.request.urlopen(url, timeout=1)
            print(f"agent card online: {url}", flush=True)
            return
        except Exception:
            time.sleep(1)
    raise RuntimeError(f"server on port {port} did not start — run it by hand to see the error")


procs = [subprocess.Popen([sys.executable, str(HERE / f)], cwd=HERE) for f in SERVERS]
try:
    for port in SERVERS.values():
        wait_for(port)
    print("\n================ plain A2A client ================", flush=True)
    subprocess.run([sys.executable, str(HERE / "a2a_client.py")], cwd=HERE, check=True)
    print("\n================ ADK orchestrator ================", flush=True)
    subprocess.run([sys.executable, str(HERE / "orchestrator.py")], cwd=HERE, check=True)
finally:
    for p in procs:
        p.terminate()

## Run the whole demo
Starts both A2A servers in the background, runs the plain client and the orchestrator, then stops the servers.

In [ ]:
!python run_demo.py

---
*SA-LIB AI Workshop 2026 · based on [build-ai-agents-free](https://github.com/Moh4696/build-ai-agents-free) by m0h (MIT)*